# 04-3. 본문 메타데이터 제거

04-2의 repeat_4gram 제외본에서 URL과 저작권·재배포 문구만 제거한다. 기자명·언론사명·사진 크레딧은 보존한다. 원본 본문은 `document_original` 열로 보존한다.

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

PROJECT = Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
assert Path(sys.prefix).resolve() == (PROJECT / ".venv").resolve(), "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."
INPUT_DIR = PROJECT / "processed_data" / "repeat_4gram_excluded"
OUTPUT_DIR = PROJECT / "processed_data" / "repeat_4gram_excluded_document_cleaned"
AUDIT_DIR = PROJECT / "review_data" / "04-3_document_metadata_cleanup"
assert INPUT_DIR.exists(), "먼저 04-2를 실행하세요."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
print("입력:", INPUT_DIR)
print("정제 데이터:", OUTPUT_DIR)
print("감사 결과:", AUDIT_DIR)

입력: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\processed_data\repeat_4gram_excluded
정제 데이터: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\processed_data\repeat_4gram_excluded_document_cleaned
감사 결과: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\review_data\04-3_document_metadata_cleanup


In [2]:
# URL과 명시적인 저작권·재배포 문구만 제거합니다. 기자명·언론사명은 보존합니다.
URL_PATTERN = re.compile(r"(?i)\b(?:https?://|www\.)[^\s<>()\[\]{}]+")
COPYRIGHT_PATTERN = re.compile(
    r"(?:무단\s*(?:전재|복제|배포)\s*(?:및\s*)?재배포\s*금지|무단\s*전재\s*금지|재배포\s*금지)",
    flags=re.IGNORECASE,
)

def clean_document(document):
    text = str(document)
    counts = {"url_removed": 0, "copyright_removed": 0}

    text, counts["url_removed"] = URL_PATTERN.subn("", text)
    text, counts["copyright_removed"] = COPYRIGHT_PATTERN.subn("", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text).strip()
    return text, counts

# 의도한 형태만 지우는지 간단한 예시로 확인합니다.
example = "서울=연합뉴스 김철수 기자 = 기사 본문 https://example.com 무단 전재 및 재배포 금지"
example_clean, example_counts = clean_document(example)
print("정제 전:", example)
print("정제 후:", example_clean)
print("제거 수:", example_counts)

정제 전: 서울=연합뉴스 김철수 기자 = 기사 본문 https://example.com 무단 전재 및 재배포 금지
정제 후: 서울=연합뉴스 김철수 기자 = 기사 본문
제거 수: {'url_removed': 1, 'copyright_removed': 1}


In [3]:
frames = []
for split in ["train", "validation", "test"]:
    frame = pd.read_csv(INPUT_DIR / f"{split}.csv").copy()
    cleaned = frame["document"].map(clean_document)
    frame.insert(frame.columns.get_loc("document") + 1, "document_original", frame["document"])
    frame["document"] = cleaned.map(lambda item: item[0])
    counts = pd.DataFrame(cleaned.map(lambda item: item[1]).tolist())
    frame = pd.concat([frame, counts], axis=1)
    frame["document_changed"] = frame["document"] != frame["document_original"]
    frames.append(frame)

all_df = pd.concat(frames, ignore_index=True)
summary = (all_df.groupby("split", sort=False).agg(
    rows=("row_id", "size"),
    changed_documents=("document_changed", "sum"),
    url_removed=("url_removed", "sum"),
    copyright_removed=("copyright_removed", "sum"),
)
    .reset_index())
display(summary)

,split,rows,changed_documents,url_removed,copyright_removed
0,train,21073,82,73,29
1,validation,2345,7,7,1
2,test,2606,15,20,0


In [4]:
# 학습용 clean split과 변경 행 감사표를 별도로 저장합니다.
for split, frame in all_df.groupby("split", sort=False):
    frame.to_csv(OUTPUT_DIR / f"{split}.csv", index=False, encoding="utf-8-sig")

audit_columns = [
    "row_id", "split", "title", "link", "document_original", "document",
    "url_removed", "copyright_removed",
]
audit = all_df.loc[all_df["document_changed"], audit_columns].copy()
audit["review_decision"] = ""
audit["memo"] = ""
audit.to_csv(AUDIT_DIR / "changed_document_comparison.csv", index=False, encoding="utf-8-sig")
summary.to_csv(AUDIT_DIR / "cleanup_summary.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([{
    "input_data": str(INPUT_DIR),
    "output_data": str(OUTPUT_DIR),
    "policy": "remove URL and explicit copyright/republication phrases only; preserve reporter and press names",
    "preserved": "document_original column retains the complete pre-cleaning text",
}]).to_csv(AUDIT_DIR / "run_info.csv", index=False, encoding="utf-8-sig")

print("저장 완료")
print("-", AUDIT_DIR / "changed_document_comparison.csv")
print("-", OUTPUT_DIR / "train.csv")

저장 완료
- C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\review_data\04-3_document_metadata_cleanup\changed_document_comparison.csv
- C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\processed_data\repeat_4gram_excluded_document_cleaned\train.csv


학습에는 `document` 열을 사용하고, 검토에는 `document_original` 열과 `changed_document_comparison.csv`를 사용한다. 감사표 표본을 확인해 바이라인 정규식이 정상 본문을 지우지 않았는지 확인한 뒤 이 버전을 채택한다.